In [1]:
!pip install -q segmentation-models-pytorch rasterio geopandas tqdm albumentations

import os
import zipfile
import glob
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.merge import merge
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from tqdm.notebook import tqdm
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Aktive Hardware: {device}")

# ZIP entpacken
zip_path = '/content/V3_Kreis11_Kreis9.zip'
extract_dir = '/content/data'
os.makedirs(extract_dir, exist_ok=True)

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    # Prüfen, ob ein Unterordner (z.B. V3_Kreis11_Kreis9/) im ZIP enthalten war
    items = [i for i in os.listdir(extract_dir) if not i.startswith('__MACOSX') and not i.startswith('.')]
    if len(items) == 1 and os.path.isdir(os.path.join(extract_dir, items[0])):
        data_dir = os.path.join(extract_dir, items[0])
    else:
        data_dir = extract_dir

    print(f"✅ Datenordner erfolgreich gesetzt auf: {data_dir}")
    print("Gefundene Dateien:", sorted([f for f in os.listdir(data_dir) if not f.startswith('.')]))
else:
    print("⚠️ Datei '/content/V3_Kreis11_Kreis9.zip' nicht gefunden. Bitte links im Datei-Tab hochladen.")

Aktive Hardware: cuda
✅ Datenordner erfolgreich gesetzt auf: /content/data
Gefundene Dateien: ['1m_Kreis11_2013.tif', '1m_Kreis11_2025.tif', '1m_Kreis9_2013.tif', '1m_Kreis9_2025.tif', 'Split_Blocs_Kreis11.gpkg', 'Split_Blocs_Kreis9_400m.gpkg', 'rasterized_1m_Kreis9_2013.tif', 'rasterized_1m_Kreis9_2025.tif', 'rasterized_Kreis11_2013_v5.tif', 'rasterized_Kreis11_2025.tif']


In [2]:
def create_chips_and_inspect(ortho_path, mask_path, grid_path, output_base_dir, dataset_name):
    out_img_dir = os.path.join(output_base_dir, dataset_name, 'images')
    out_mask_dir = os.path.join(output_base_dir, dataset_name, 'masks')
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)

    grid = gpd.read_file(grid_path)
    valid_tiles_count = 0
    unique_mask_values = set()

    with rasterio.open(ortho_path) as src_img, rasterio.open(mask_path) as src_mask:
        meta_img = src_img.meta.copy()
        meta_mask = src_mask.meta.copy()

        meta_img.update({"driver": "GTiff", "height": 400, "width": 400})
        meta_mask.update({"driver": "GTiff", "height": 400, "width": 400})

        for idx, geom in enumerate(grid.geometry):
            try:
                out_img, transform_img = mask(src_img, [geom], crop=True)
                out_lbl, transform_mask = mask(src_mask, [geom], crop=True)

                _, h, w = out_img.shape
                fixed_img = np.zeros((src_img.count, 400, 400), dtype=out_img.dtype)
                fixed_lbl = np.full((1, 400, 400), 255, dtype=out_lbl.dtype)

                copy_h = min(h, 400)
                copy_w = min(w, 400)

                fixed_img[:, :copy_h, :copy_w] = out_img[:, :copy_h, :copy_w]
                fixed_lbl[:, :copy_h, :copy_w] = out_lbl[:, :copy_h, :copy_w]

                # Leere Rand-Kacheln ignorieren
                if np.all((fixed_lbl == -1) | (fixed_lbl == 255)):
                    continue

                unique_mask_values.update(np.unique(fixed_lbl).tolist())

                meta_img.update({"transform": transform_img})
                meta_mask.update({"transform": transform_mask})

                img_filename = os.path.join(out_img_dir, f"{dataset_name}_tile_{idx}.tif")
                mask_filename = os.path.join(out_mask_dir, f"{dataset_name}_tile_{idx}.tif")

                with rasterio.open(img_filename, "w", **meta_img) as dest_img:
                    dest_img.write(fixed_img)
                with rasterio.open(mask_filename, "w", **meta_mask) as dest_mask:
                    dest_mask.write(fixed_lbl)

                valid_tiles_count += 1
            except ValueError:
                continue

    print(f"✅ {dataset_name}: {valid_tiles_count} Kacheln gespeichert | Masken-Werte: {sorted(list(unique_mask_values))}")

# --- AUSFÜHRUNG ---
dataset_dir = '/content/dataset'
grid_k11 = os.path.join(data_dir, 'Split_Blocs_Kreis11.gpkg')
grid_k9 = os.path.join(data_dir, 'Split_Blocs_Kreis9_400m.gpkg')

create_chips_and_inspect(os.path.join(data_dir, '1m_Kreis9_2013.tif'), os.path.join(data_dir, 'rasterized_1m_Kreis9_2013.tif'), grid_k9, dataset_dir, 'Kreis9_2013')
create_chips_and_inspect(os.path.join(data_dir, '1m_Kreis9_2025.tif'), os.path.join(data_dir, 'rasterized_1m_Kreis9_2025.tif'), grid_k9, dataset_dir, 'Kreis9_2025')
create_chips_and_inspect(os.path.join(data_dir, '1m_Kreis11_2013.tif'), os.path.join(data_dir, 'rasterized_Kreis11_2013_v5.tif'), grid_k11, dataset_dir, 'Kreis11_2013')
create_chips_and_inspect(os.path.join(data_dir, '1m_Kreis11_2025.tif'), os.path.join(data_dir, 'rasterized_Kreis11_2025.tif'), grid_k11, dataset_dir, 'Kreis11_2025')

✅ Kreis9_2013: 92 Kacheln gespeichert | Masken-Werte: [0, 1, 255]
✅ Kreis9_2025: 92 Kacheln gespeichert | Masken-Werte: [0.0, 1.0, 255.0]
✅ Kreis11_2013: 110 Kacheln gespeichert | Masken-Werte: [0, 1, 255]
✅ Kreis11_2025: 110 Kacheln gespeichert | Masken-Werte: [0, 1, 255]


In [3]:
# ImageNet-Normalisierung & Augmentation für besseres Spatial Transfer
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

class SealedSurfaceDataset(Dataset):
    def __init__(self, folder_list, transform=None):
        self.img_paths = []
        self.mask_paths = []
        self.transform = transform
        for folder in folder_list:
            imgs = sorted(glob.glob(os.path.join(folder, 'images', '*.tif')))
            for img_p in imgs:
                mask_p = img_p.replace('/images/', '/masks/')
                if os.path.exists(mask_p):
                    self.img_paths.append(img_p)
                    self.mask_paths.append(mask_p)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.img_paths[idx]) as src:
            # Albumentations erwartet (H, W, C) als uint8
            img = np.transpose(src.read()[:3], (1, 2, 0)).astype(np.uint8)
        with rasterio.open(self.mask_paths[idx]) as src:
            mask_raw = src.read(1).astype(np.int64)

        if self.transform:
            augmented = self.transform(image=img, mask=mask_raw)
            img = augmented['image']
            mask_raw = augmented['mask']

        # Zurück zu PyTorch (C, H, W)
        img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(mask_raw).long()
        return img_tensor, mask_tensor

# Trainingsdaten: Strikt nur Kreis 9 (184 Kacheln total)
train_folders = [
    os.path.join(dataset_dir, 'Kreis9_2013'),
    os.path.join(dataset_dir, 'Kreis9_2025')
]
train_dataset = SealedSurfaceDataset(train_folders, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, drop_last=True)

# U-Net mit ResNet34
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=2
).to(device)

# Kombinierter Loss mit striktem ignore_index=255
ce_loss = nn.CrossEntropyLoss(ignore_index=255)
dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=255)

def combined_loss(pred, target):
    return 0.5 * ce_loss(pred, target) + 0.5 * dice_loss(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

epochs = 25
print(f"Starte optimiertes Training auf {len(train_dataset)} Kreis-9-Kacheln für {epochs} Epochen...")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoche {epoch+1:02d}/{epochs:02d} | Loss: {epoch_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), '/content/unet_resnet34_kreis9_v2.pth')
print("✅ Optimiertes Modell gespeichert!")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Starte optimiertes Training auf 184 Kreis-9-Kacheln für 25 Epochen...
Epoche 01/25 | Loss: 0.4921
Epoche 05/25 | Loss: 0.3019
Epoche 10/25 | Loss: 0.2588
Epoche 15/25 | Loss: 0.2562
Epoche 20/25 | Loss: 0.2280
Epoche 25/25 | Loss: 0.2260
✅ Optimiertes Modell gespeichert!


In [4]:
dir_2013 = os.path.join(dataset_dir, 'Kreis11_2013', 'images')
dir_2025 = os.path.join(dataset_dir, 'Kreis11_2025', 'images')
mask_dir_2013 = os.path.join(dataset_dir, 'Kreis11_2013', 'masks')
mask_dir_2025 = os.path.join(dataset_dir, 'Kreis11_2025', 'masks')

model.eval()

# ImageNet-Normalisierung identisch zum Training
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def evaluate_year_metrics(img_dir, mask_dir, year):
    tp, fp, fn, pred_area, ref_area = 0, 0, 0, 0, 0
    files_list = sorted([f for f in os.listdir(img_dir) if f.endswith('.tif')])

    with torch.no_grad():
        for f in files_list:
            img_p = os.path.join(img_dir, f)
            mask_p = os.path.join(mask_dir, f)
            if not os.path.exists(mask_p):
                continue

            with rasterio.open(img_p) as src:
                img = src.read()[:3].astype(np.float32) / 255.0
            with rasterio.open(mask_p) as src:
                mask_raw = src.read(1).astype(np.int64)

            # Normalisieren & Vorhersage berechnen
            tensor_img = torch.from_numpy(img).unsqueeze(0).to(device)
            tensor_img = (tensor_img - mean) / std
            pred = torch.argmax(model(tensor_img), dim=1).squeeze()

            m_tensor = torch.from_numpy(mask_raw).to(device)

            # Nur gültige Pixel innerhalb der Bezirksgrenze (0 oder 1, ohne 255 NoData) auswerten
            valid_mask = (m_tensor != 255)
            p_valid = pred[valid_mask]
            m_valid = m_tensor[valid_mask]

            pred_area += (p_valid == 1).sum().item()
            ref_area += (m_valid == 1).sum().item()

            tp += torch.logical_and(p_valid == 1, m_valid == 1).sum().item()
            fp += torch.logical_and(p_valid == 1, m_valid == 0).sum().item()
            fn += torch.logical_and(p_valid == 0, m_valid == 1).sum().item()

    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)

    print(f"--- Evaluierung Kreis 11 ({year}) ---")
    print(f"Precision (Sealed): {precision:.4f}")
    print(f"Recall (Sealed):    {recall:.4f}")
    print(f"F1 / Dice Score:    {f1:.4f}")
    print(f"IoU (Sealed):       {iou:.4f}")
    print(f"Predicted Area:     {pred_area:,.0f} m² | Reference Area: {ref_area:,.0f} m²\n")

evaluate_year_metrics(dir_2013, mask_dir_2013, "2013")
evaluate_year_metrics(dir_2025, mask_dir_2025, "2025")

--- Evaluierung Kreis 11 (2013) ---
Precision (Sealed): 0.9217
Recall (Sealed):    0.8693
F1 / Dice Score:    0.8947
IoU (Sealed):       0.8095
Predicted Area:     7,858,249 m² | Reference Area: 8,332,101 m²

--- Evaluierung Kreis 11 (2025) ---
Precision (Sealed): 0.9197
Recall (Sealed):    0.8619
F1 / Dice Score:    0.8899
IoU (Sealed):       0.8016
Predicted Area:     7,697,113 m² | Reference Area: 8,212,913 m²



In [5]:
output_tmp_dir = '/content/Oerlikon_V2_Output'
os.makedirs(output_tmp_dir, exist_ok=True)
src_files_13, src_files_25 = [], []

sample_files = sorted([f for f in os.listdir(dir_2013) if f.endswith('.tif')])
model.eval()

for file_13 in tqdm(sample_files, desc="Generiere V2 Probability-Kacheln"):
    file_25 = file_13.replace('2013', '2025')
    p13 = os.path.join(dir_2013, file_13)
    p25 = os.path.join(dir_2025, file_25)
    m13 = os.path.join(mask_dir_2013, file_13)
    m25 = os.path.join(mask_dir_2025, file_25)

    if not os.path.exists(p25):
        continue

    with rasterio.open(p13) as s13, rasterio.open(p25) as s25:
        meta = s13.meta.copy()
        img_13 = s13.read()[:3].astype(np.float32) / 255.0
        img_25 = s25.read()[:3].astype(np.float32) / 255.0

    # Valid-Maske laden, um gepolsterte Ränder (255) sauber als NoData (-9999) zu setzen
    with rasterio.open(m13) as sm13, rasterio.open(m25) as sm25:
        valid_13 = (sm13.read(1) != 255)
        valid_25 = (sm25.read(1) != 255)

    t13 = (torch.from_numpy(img_13).unsqueeze(0).to(device) - mean) / std
    t25 = (torch.from_numpy(img_25).unsqueeze(0).to(device) - mean) / std

    with torch.no_grad():
        prob_13 = torch.softmax(model(t13), dim=1)[:, 1, :, :].squeeze().cpu().numpy()
        prob_25 = torch.softmax(model(t25), dim=1)[:, 1, :, :].squeeze().cpu().numpy()

    # Ränder ausserhalb von Kreis 11 auf -9999 setzen
    prob_13 = np.where(valid_13, prob_13, -9999.0).astype(np.float32)
    prob_25 = np.where(valid_25, prob_25, -9999.0).astype(np.float32)

    meta.update({'count': 1, 'dtype': 'float32', 'nodata': -9999.0})
    t_path_13 = os.path.join(output_tmp_dir, f"v2_prob13_{file_13}")
    t_path_25 = os.path.join(output_tmp_dir, f"v2_prob25_{file_25}")

    with rasterio.open(t_path_13, 'w', **meta) as dst:
        dst.write(prob_13, 1)
    with rasterio.open(t_path_25, 'w', **meta) as dst:
        dst.write(prob_25, 1)

    src_files_13.append(t_path_13)
    src_files_25.append(t_path_25)

def build_mosaic(file_list, out_name):
    opened = [rasterio.open(fp) for fp in file_list]
    mosaic, out_trans = merge(opened, nodata=-9999.0)
    out_meta = opened[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": -9999.0
    })

    out_path = os.path.join('/content', out_name)
    with rasterio.open(out_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    for f in opened:
        f.close()
    return out_path

mosaic_13_path = build_mosaic(src_files_13, 'Oerlikon_Prob_2013_V2.tif')
mosaic_25_path = build_mosaic(src_files_25, 'Oerlikon_Prob_2025_V2.tif')

print("\n✅ Neue V2-Mosaike erstellt! Download startet...")
files.download(mosaic_13_path)
files.download(mosaic_25_path)

Generiere V2 Probability-Kacheln:   0%|          | 0/110 [00:00<?, ?it/s]


✅ Neue V2-Mosaike erstellt! Download startet...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>